# GPU run (Kaggle)
Settings: **Accelerator = GPU T4 x2 (or P100)**, **Internet = On**. Upload the project as a Kaggle Dataset named `genai-a2` (zip of the project folder without `checkpoints/base`, `data/raw`), or `git clone` your repo in cell 1.

In [ ]:
import os, shutil, glob, zipfile
# 1) find the project anywhere under /kaggle/input (handles different mount paths and un-extracted zips)
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall('/kaggle/working/unz')
hits = [os.path.dirname(os.path.dirname(f)) for f in glob.glob('/kaggle/input/**/scripts/kaggle_pipeline.py', recursive=True)
        + glob.glob('/kaggle/working/unz/**/scripts/kaggle_pipeline.py', recursive=True)]
print('found project at:', hits)
assert hits, "Project not found. Right panel -> Input -> Add Input -> attach your genai-a2 dataset, then re-run."
shutil.copytree(hits[0], '/kaggle/working/proj', dirs_exist_ok=True)
%cd /kaggle/working/proj
!ls

In [ ]:
!pip -q install peft psutil fastapi uvicorn
!nvidia-smi | head -15
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# Full pipeline: train 1.7B target + 360M draft, evaluate, benchmark (about 1-2 h on a T4)
!python scripts/kaggle_pipeline.py

In [ ]:
# Package outputs: download /kaggle/working/gpu_results.zip from the Output tab and unzip into the project on your PC
!zip -r /kaggle/working/gpu_results.zip results/metrics_gpu results/logs checkpoints/adapters data/processed -x '*/optimizer*' '*/epoch*/*.safetensors'
!ls -lh /kaggle/working/gpu_results.zip